In [ ]:
import json
with open('tnn_lists.json', 'r') as file:
    data = json.load(file)
data_final=[]
for i in range(20000):
    data_final.append(data[i])


In [ ]:
import requests
selected_data=[]
for i in range(2000):
    source=requests.get(data_final[i])
    a=source.json()
    data=json.loads(a)
    for i in data:
        selected={'Title': i.get('title'), 'Text': i.get('text'), 'Synopsis': i.get('synopsis'), 'Insert Date': i.get('insertdate'), 'Author': i.get('authors'), 'Keywords': i.get('keywords')}
        selected_data.append(selected)

ConnectionError: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))

In [27]:
import requests
import json
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Setup retry-enabled session
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retries)
session.mount('http://', adapter)
session.mount('https://', adapter)

selected_data = []

for i in range(len(data_final)):  # or limit to range(20000)
    try:
        response = session.get(data_final[i], timeout=10)
        a = response.json()
        
        # No need for json.loads() here; response.json() already parses it
        for item in a:
            if isinstance(item, str):
                item = item.strip()
            if not item:
                continue
            try:
                item = json.loads(item)
            except json.JSONDecodeError:
                #print(f"Skipping bad item: {item}")
                continue
            if not isinstance(item, dict):
                #print(f"Skipping non-dict item: {item}")
                continue
            selected = {
                'Title': item.get('title'),
                'Text': item.get('text'),
                'Synopsis': item.get('synopsis'),
                'Insert Date': item.get('insertdate'),
                'Author': item.get('authors')
            }
            selected_data.append(selected)

        time.sleep(0.5)  # polite delay to avoid hammering the server

    except requests.exceptions.RequestException as e:
        print(f"Request failed at index {i}: {e}")
        continue


Request failed at index 5587: HTTPSConnectionPool(host='times-network.s3.ap-southeast-1.amazonaws.com', port=443): Max retries exceeded with url: /article-content/timesnownews/DR_TN_108272051.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x108cc6fd0>: Failed to resolve 'times-network.s3.ap-southeast-1.amazonaws.com' ([Errno 8] nodename nor servname provided, or not known)"))
Request failed at index 15918: HTTPSConnectionPool(host='times-network.s3.ap-southeast-1.amazonaws.com', port=443): Read timed out.
Request failed at index 16113: HTTPSConnectionPool(host='times-network.s3.ap-southeast-1.amazonaws.com', port=443): Max retries exceeded with url: /article-content/timesnownews/DR_TN_108397806.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x109e65f90>: Failed to resolve 'times-network.s3.ap-southeast-1.amazonaws.com' ([Errno 8] nodename nor servname provided, or not known)"))
Request failed at index 16114: HTTPSC

KeyboardInterrupt: 